# 10 · Files, JSON, CSV and pathlib

After this notebook you can walk folders with `pathlib`, read and write text without encoding bugs, stream files bigger than memory, and move data between JSON, JSONL and CSV. That is the plumbing under every RAG ingestion job and every fine-tuning dataset.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [09 · Errors and exceptions](09_errors_and_exceptions.ipynb)

## Why this matters in interviews

- Take-home and live-coding rounds for AI engineers usually start with "here is a folder of documents" or "here is a CSV of Q&A pairs". Loading it cleanly, with sources kept, is table stakes.
- JSONL is the file format of fine-tuning datasets, eval sets, batch-API inputs and request logs. You should be able to write and validate it from memory.
- "The file is 5 GB and you have 2 GB of RAM" is a classic memory question. The answer is streaming, and in Python that means iterating the file (a generator).
- Encoding and path bugs (Windows' non-UTF-8 default, string-glued paths, `../` escaping a sandbox) are cheap points for the interviewer to probe.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `dt01` | You need to process a 5 GB CSV on a server with 2 GB of RAM. What do you do? |
| `dt04` | What is a generator in Python and why does it matter here? |
| `dt07` | How would you build the ingestion pipeline for a RAG system? (the loading step) |
| `dt14` | How do you build a training dataset from production logs? (the JSONL step) |

In [ ]:
import csv
import json
import locale
import random
import shutil
import tempfile
import tracemalloc
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True})

WORK = Path(tempfile.mkdtemp(prefix="nb10_"))    # all scratch files live here, never in the repo
print("scratch folder:", WORK.name)

## 1. `pathlib`: paths are objects, not strings

**In plain English:** a `Path` knows which parts are the folder, the file name and the extension, and joins pieces with `/` using the right separator for your OS. No more `os.path.join` chains or bugs from gluing strings with `"\\"`.

**Predict:** for `data/raw/reviews.v2.jsonl`, what are `.stem` and `.suffix`? (`PurePosixPath` just keeps the printout identical on Windows, macOS and Linux.)

In [ ]:
p = PurePosixPath("data/raw/reviews.v2.jsonl")
print("name       :", p.name)
print("stem       :", p.stem)
print("suffix     :", p.suffix)
print("suffixes   :", p.suffixes)
print("parent     :", p.parent)
print("parts      :", p.parts)
print("with_suffix:", p.with_suffix(".csv"))
print("joined     :", PurePosixPath("data") / "processed" / "reviews.parquet")
assert p.stem == "reviews.v2" and p.suffix == ".jsonl"

`.stem` drops only the **last** extension. Now let's make a real folder tree — a tiny knowledge base we will load for RAG later. It deliberately includes the awkward cases real folders have: an empty file, a hidden draft, a non-Markdown file, and a legacy file that is not UTF-8.

In [ ]:
KB = WORK / "kb"
files = {
    "policies/refunds.md":  "# Refunds\n\nRefunds are issued within 30 days of delivery. Digital goods are non-refundable.",
    "policies/shipping.md": "# Shipping\n\nStandard shipping takes 3-5 business days. Express shipping takes 1-2 days.",
    "faq/account.md":       "# Account\n\nYou can reset your password from the login page. Two-factor login is optional.",
    "faq/empty.md":         "   \n",
    "notes/.draft.md":      "# Draft\n\nNot ready for customers.",
    "README.txt":           "Not Markdown - a .md loader should ignore me.",
}
for rel, text in files.items():
    path = KB / rel
    path.parent.mkdir(parents=True, exist_ok=True)       # like `mkdir -p`
    path.write_text(text, encoding="utf-8")
(KB / "faq" / "legacy.md").write_bytes("# Café hours\n\nThe café opens at 8am.".encode("cp1252"))  # not UTF-8!

print("glob('*')     :", sorted(p.name for p in KB.glob("*")))
print("glob('*.md')  :", sorted(p.name for p in KB.glob("*.md")), "<- top level only")
print("rglob('*.md') :", sorted(p.relative_to(KB).as_posix() for p in KB.rglob("*.md")))
refunds = KB / "policies" / "refunds.md"
print("exists / is_file / size:", refunds.exists(), refunds.is_file(), refunds.stat().st_size, "bytes")

`glob` looks in one folder; `rglob` (same as `glob("**/...")`) looks recursively. Two things to notice: pathlib's `*` **does** match hidden names like `.draft.md` (unlike your shell), and the order `glob` returns is up to the filesystem — so always `sorted(...)` when you need reproducible output.

**A security detail for agents with file tools.** If a model (or a user) supplies a path, `base / user_path` can escape your folder: `../` walks up, and an absolute path replaces the base entirely. Resolve it and check it is still inside:

In [ ]:
def safe_join(base, user_path):
    target = (base / user_path).resolve()
    if not target.is_relative_to(base.resolve()):
        raise ValueError(f"path escapes the sandbox: {user_path!r}")
    return target

print("ok     :", safe_join(KB, "policies/refunds.md").relative_to(KB.resolve()).as_posix())
for attempt in ["../../secrets.txt", "/etc/passwd"]:
    try:
        safe_join(KB, attempt)
    except ValueError as e:
        print("blocked:", e)

## 2. Reading and writing text — always pass `encoding="utf-8"`

Without an `encoding=` argument, `open()`, `read_text()` and `write_text()` use your **locale's** encoding. On Linux and macOS that is usually UTF-8; on many Windows machines it is `cp1252`. Python 3.15 is slated to make UTF-8 the default everywhere (PEP 686). Until then, the same code can work on your laptop and corrupt text on a colleague's machine. A wrong encoding does not always raise an error — sometimes it quietly produces garbage ("mojibake"):

In [ ]:
note = WORK / "note.txt"
original = "café — naïve résumé 👋"
note.write_text(original, encoding="utf-8")

print("characters / bytes on disk:", len(original), "/", len(note.read_bytes()))
print("read as utf-8 :", note.read_text(encoding="utf-8"))
mojibake = note.read_text(encoding="cp1252")
print("read as cp1252:", mojibake, " <- no error, just wrong")
print("this machine's default encoding:", locale.getpreferredencoding(False))

try:
    (KB / "faq" / "legacy.md").read_text(encoding="utf-8")      # cp1252 bytes read as UTF-8
except UnicodeDecodeError as e:
    print("legacy.md as utf-8 ->", type(e).__name__ + ":", e.reason, f"(byte {e.object[e.start]:#x})")
assert mojibake != original

## 3. `open()` modes and `with`

| Mode | Meaning | Watch out |
|---|---|---|
| `"r"` | read text (default) | `FileNotFoundError` if missing |
| `"w"` | write text | **truncates** an existing file to empty first |
| `"a"` | append text | creates the file if missing; perfect for logs and JSONL |
| `"x"` | create, fail if it exists | safe "never overwrite" |
| `"rb"` / `"wb"` | bytes (images, PDFs, pickles) | no `encoding=` in binary mode |
| `"r+"` | read and write | rarely what you want |

`with open(...) as f:` closes the file when the block ends — even if an exception is raised inside. For CSV files also pass `newline=""` (section 7 explains why).

In [ ]:
run_log = WORK / "run.log"
with open(run_log, "w", encoding="utf-8") as handle:     # "w" creates or truncates
    handle.write("run 1\n")
with open(run_log, "a", encoding="utf-8") as f:          # "a" appends
    f.write("run 2\n")
print("lines:", run_log.read_text(encoding="utf-8").splitlines())
print("closed after the with block:", handle.closed)

try:
    with open(run_log, "x", encoding="utf-8") as f:      # "x" refuses to overwrite
        f.write("oops")
except FileExistsError as e:
    print("mode 'x' ->", type(e).__name__)

## 4. Big files: read lazily, one line at a time

`path.read_text()` or `f.read()` loads the **whole file** into memory. Iterating the file object — `for line in f:` — hands you one line at a time, so memory stays flat however big the file is. The size of the file is never the constraint; the largest object you keep alive is.

Let's write a 50,000-line request log in JSONL and compare peak memory (measured with `tracemalloc`) for the two ways of totalling its tokens.

In [ ]:
rng = random.Random(0)
requests_log = WORK / "requests.jsonl"
with open(requests_log, "w", encoding="utf-8") as f:
    for i in range(50_000):
        rec = {"id": i, "model": rng.choice(["gpt-4.1-mini", "gpt-4.1-nano"]),
               "latency_ms": rng.randint(200, 3000), "tokens": rng.randint(50, 1500)}
        f.write(json.dumps(rec) + "\n")
size_mb = requests_log.stat().st_size / 1e6
print(f"requests.jsonl: {size_mb:.1f} MB")

def tokens_all_at_once(path):
    lines = path.read_text(encoding="utf-8").splitlines()     # the whole file in memory
    return sum(json.loads(line)["tokens"] for line in lines)

def tokens_streaming(path):
    with open(path, encoding="utf-8") as f:                   # one line at a time
        return sum(json.loads(line)["tokens"] for line in f)

def peak_mb(fn, *args):
    tracemalloc.start()
    result = fn(*args)
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return result, peak / 1e6

total_a, peak_a = peak_mb(tokens_all_at_once, requests_log)
total_b, peak_b = peak_mb(tokens_streaming, requests_log)
print(f"all at once: total={total_a:,}  peak memory {peak_a * 1000:8,.0f} KB")
print(f"streaming  : total={total_b:,}  peak memory {peak_b * 1000:8,.0f} KB")
assert total_a == total_b and peak_b < peak_a / 10

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
bars = ax.barh(["streaming\nfor line in f", "all at once\nread_text()"], [peak_b, peak_a], color=[BLUE, ORANGE], height=0.55)
ax.bar_label(bars, labels=[f"{peak_b * 1000:.0f} KB", f"{peak_a:.1f} MB"], padding=4)
ax.set_xlabel("peak Python memory while summing tokens (MB)")
ax.set_title(f"Same {size_mb:.1f} MB file, same total: memory depends on what you hold")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, peak_a * 1.2)
plt.show()

Scale the file to 5 GB and the streaming version still needs about the same few kilobytes, while the all-at-once version needs several times the file size. The same idea gives you **generator pipelines**: each stage pulls one item from the stage before, and nothing is read until the final consumer asks. The counter below proves that nothing is read until we ask for a result.

In [ ]:
lines_read = 0

def read_jsonl(path):
    global lines_read
    with open(path, encoding="utf-8") as f:
        for line in f:
            lines_read += 1
            if line.strip():
                yield json.loads(line)

def slow(records, threshold_ms):
    return (r for r in records if r["latency_ms"] > threshold_ms)

pipeline = slow(read_jsonl(requests_log), threshold_ms=2900)
print("lines read after building the pipeline:", lines_read)
first = next(pipeline)
print("lines read to find the first slow request:", lines_read, "->", first)
remaining = sum(1 for _ in pipeline)
print(f"slow requests in the rest of the file: {remaining}  (lines read: {lines_read:,})")
print("second pass over the same generator:", sum(1 for _ in pipeline), "<- generators are single-use")
assert lines_read == 50_000

## 5. JSON: `dumps` / `loads`, and what does not survive the round trip

| Python | JSON | Back in Python |
|---|---|---|
| `dict` | object | `dict` — but **keys become strings** |
| `list`, `tuple` | array | `list` (tuples come back as lists) |
| `str`, `int`, `float` | string, number | same |
| `True` / `False` / `None` | `true` / `false` / `null` | same |
| `datetime`, `set`, `Path`, `Decimal` | — | `TypeError` unless you pass `default=` |

`json.dumps`/`json.loads` work on strings; `json.dump(obj, f)`/`json.load(f)` work on open files.

**Predict:** is the round trip below equal to the original?

In [ ]:
record = {"id": 7, "question": "¿Dónde está mi pedido?", "tags": ("shipping", "es"), "score": None, 1: "int key"}
text = json.dumps(record)
back = json.loads(text)
print("dumps      :", text)
print("loads      :", back)
print("equal?     :", back == record)
print("readable   :", json.dumps(record, ensure_ascii=False))
print("pretty     :\n" + json.dumps({"model": "gpt-4.1-mini", "params": {"temperature": 0}}, indent=2))
assert back["tags"] == ["shipping", "es"] and "1" in back

Three options you will use constantly:

- `ensure_ascii=False` keeps non-English text readable (and smaller) instead of `¿` escapes.
- `default=fn` teaches `dumps` about types it does not know, like `datetime` or `set`.
- `sort_keys=True` gives the same text for the same data regardless of key order — needed when you hash a request to use as a cache key.

Also: `json.dumps(float("nan"))` returns `NaN`, which is **not valid JSON** — strict parsers (and APIs) reject it. Pass `allow_nan=False` to fail early instead.

In [ ]:
def to_jsonable(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, (set, frozenset)):
        return sorted(obj)
    if isinstance(obj, Path):
        return obj.as_posix()
    raise TypeError(f"{type(obj).__name__} is not JSON serializable")

event = {"at": datetime(2026, 9, 29, 10, 30, tzinfo=timezone.utc), "labels": {"refund", "angry"}}
try:
    json.dumps(event)
except TypeError as e:
    print("without default:", e)
print("with default   :", json.dumps(event, default=to_jsonable))
print("key order      :", json.dumps({"b": 1, "a": 2}) == json.dumps({"a": 2, "b": 1}),
      "| sort_keys:", json.dumps({"b": 1, "a": 2}, sort_keys=True) == json.dumps({"a": 2, "b": 1}, sort_keys=True))
print("NaN            :", json.dumps({"score": float("nan")}), "<- not valid JSON")
try:
    json.dumps({"score": float("nan")}, allow_nan=False)
except ValueError as e:
    print("allow_nan=False:", e)

How much does `ensure_ascii` matter? For English, nothing. For other scripts each character becomes a 6-byte `\uXXXX` escape, which inflates files and — if you paste JSON into a prompt — tokens.

In [ ]:
questions = {"English": "Where is my order and when will it arrive?",
             "Spanish": "¿Dónde está mi pedido y cuándo llegará?",
             "Hindi":   "मेरा ऑर्डर कहाँ है और यह कब पहुँचेगा?",
             "Japanese": "注文した商品はどこにありますか？いつ届きますか？"}
sizes = {lang: (len(json.dumps({"q": q}).encode("utf-8")), len(json.dumps({"q": q}, ensure_ascii=False).encode("utf-8")))
         for lang, q in questions.items()}
for lang, (esc, raw) in sizes.items():
    print(f"{lang:<9} ensure_ascii=True: {esc:4d} bytes   ensure_ascii=False: {raw:4d} bytes")

fig, ax = plt.subplots(figsize=(8, 3.4))
x = range(len(sizes))
b1 = ax.bar([i - 0.2 for i in x], [s[0] for s in sizes.values()], width=0.38, color=ORANGE, label="ensure_ascii=True (default)")
b2 = ax.bar([i + 0.2 for i in x], [s[1] for s in sizes.values()], width=0.38, color=BLUE, label="ensure_ascii=False")
ax.bar_label(b1, padding=2, fontsize=8); ax.bar_label(b2, padding=2, fontsize=8)
ax.set_xticks(list(x), list(sizes)); ax.set_ylabel("bytes of JSON (UTF-8)"); ax.grid(axis="x", visible=False)
ax.set_title("The same one-question record, serialised two ways"); ax.legend(frameon=False)
plt.show()
assert sizes["Hindi"][1] < sizes["Hindi"][0] and sizes["English"][0] == sizes["English"][1]

## 6. JSONL: one JSON object per line

JSONL (also `.ndjson`) is the workhorse format for LLM data: OpenAI fine-tuning and batch files, eval sets, request logs. Why not one big JSON array?

- **Streamable:** read one record at a time (section 4). A JSON array must be parsed whole.
- **Appendable:** logging one more record is one `write`. An array needs rewriting to add the closing `]`.
- **Robust:** one corrupt line (a crash mid-write) loses one record, not the file.
- **Splittable:** shard or sample by lines with standard tools.

A good reader skips blank lines and reports bad ones with their line numbers instead of dying:

In [ ]:
def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_jsonl_report(path):
    good, bad = [], []
    with open(path, encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                good.append(json.loads(line))
            except json.JSONDecodeError as e:
                bad.append((lineno, e.msg))
    return good, bad

evals = WORK / "evals.jsonl"
write_jsonl(evals, [{"q": "2+2?", "expected": "4"}, {"q": "Capital of France?", "expected": "Paris"},
                    {"q": "¿Hola?", "expected": "Hola"}])
with open(evals, "a", encoding="utf-8") as f:
    f.write('\n{"q": "half-written rec')          # a blank line, then a record cut off by a crash
good, bad = read_jsonl_report(evals)
print(f"{len(good)} good records; bad lines: {bad}")
assert len(good) == 3 and bad[0][0] == 5

## 7. CSV: `DictReader` and `DictWriter`

CSV looks trivial until a field contains a comma, a quote or a newline. The `csv` module quotes and unquotes these correctly; `line.split(",")` does not. Two rules:

- open CSV files with `newline=""` so the `csv` module controls line endings (otherwise you get blank rows between records on Windows, and newlines inside quoted fields can break);
- everything you read back is a **string** — convert numbers yourself.

In [ ]:
rows = [
    {"question": "What is RAG?", "answer": 'Retrieval-augmented generation: "look it up, then answer".', "votes": 12},
    {"question": "Name two vector stores, please", "answer": "FAISS, pgvector", "votes": 7},
    {"question": "Can an answer span lines?", "answer": "Yes.\nThe csv module quotes it.", "votes": 3},
]
qa_csv = WORK / "qa.csv"
with open(qa_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "answer", "votes"])
    writer.writeheader()
    writer.writerows(rows)
print(qa_csv.read_text(encoding="utf-8"))

naive = [line.split(",") for line in qa_csv.read_text(encoding="utf-8").splitlines()[1:]]
print("naive split: fields per line =", [len(parts) for parts in naive], "(expected 3 each)")

with open(qa_csv, encoding="utf-8", newline="") as f:
    back = list(csv.DictReader(f))
print("DictReader : fields per row  =", [len(r) for r in back], "| votes come back as", repr(back[0]["votes"]))
assert [r["answer"] for r in back] == [r["answer"] for r in rows] and back[0]["votes"] == "12"

For a 5 GB CSV, `csv.DictReader` over an open file streams exactly like section 4 — aggregate inside the loop and keep only the running totals. (`pandas.read_csv(chunksize=...)` is the vectorised version, covered in [pandas essentials](../04_numpy_pandas_pytorch/02_pandas_essentials.ipynb).)

## 8. `tempfile` and `shutil`

- `tempfile.mkdtemp()` makes a fresh folder you clean up yourself (this notebook's `WORK`); `tempfile.TemporaryDirectory()` is a context manager that deletes the folder when the block ends. Both are the right place for scratch files in tests and notebooks.
- `shutil` does the file-manager jobs: `copy2` (copy a file with metadata), `copytree`, `move`, `rmtree` (delete a folder tree), `make_archive` (zip a folder), `which` (find an executable), `disk_usage`.

In [ ]:
with tempfile.TemporaryDirectory(prefix="nb10_tmp_") as tmp:
    tmp = Path(tmp)
    (tmp / "scratch.txt").write_text("temporary", encoding="utf-8")
    print("inside the block:", [p.name for p in tmp.iterdir()])
print("after the block, folder exists?", tmp.exists())

backup = shutil.copytree(KB, WORK / "kb_backup")                     # copy the whole tree
archive = shutil.make_archive(str(WORK / "kb_snapshot"), "zip", root_dir=KB)
print("backup has", sum(1 for p in backup.rglob("*") if p.is_file()), "files; archive:", Path(archive).name,
      f"({Path(archive).stat().st_size} bytes)")
shutil.rmtree(backup)
print("backup removed:", not backup.exists())

## 9. Practical A: load a folder of Markdown docs for RAG

The first step of any RAG ingestion: turn a folder into a list of records, each carrying its **source** (so answers can cite it later) and its text. A careful loader:

- walks the tree with `rglob`, **sorted** so runs are reproducible;
- skips hidden files and empty files, and says so;
- survives a file that is not UTF-8 rather than crashing the whole job — and records the fallback it used;
- keeps the path relative to the root in POSIX form, so the `source` is the same on every OS.

In [ ]:
def load_markdown_docs(root):
    records, issues = [], []
    for path in sorted(root.rglob("*.md")):
        rel = path.relative_to(root)
        source = rel.as_posix()
        if any(part.startswith(".") for part in rel.parts):
            issues.append((source, "hidden: skipped"))
            continue
        try:
            text = path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            text = path.read_text(encoding="cp1252")      # legacy Windows file; flag it for a human
            issues.append((source, "not UTF-8: decoded as cp1252"))
        text = text.strip()
        if not text:
            issues.append((source, "empty: skipped"))
            continue
        title = next((line.lstrip("#").strip() for line in text.splitlines() if line.startswith("#")), path.stem)
        records.append({"source": source, "title": title, "text": text})
    return records, issues

docs, issues = load_markdown_docs(KB)
for d in docs:
    print(f"{d['source']:<22} {d['title']!r:<16} {len(d['text']):3d} chars")
print("issues:", issues)
assert [d["source"] for d in docs] == ["faq/account.md", "faq/legacy.md", "policies/refunds.md", "policies/shipping.md"]
assert docs[1]["title"] == "Café hours"

Save the records as JSONL — the hand-off format to the chunking and embedding steps — and check the round trip is lossless:

In [ ]:
kb_jsonl = WORK / "kb_records.jsonl"
write_jsonl(kb_jsonl, docs)
reloaded, bad = read_jsonl_report(kb_jsonl)
print(kb_jsonl.read_text(encoding="utf-8").splitlines()[0][:110] + " ...")
assert reloaded == docs and bad == []
print(f"{len(reloaded)} records round-tripped through JSONL unchanged")

Next in a real pipeline: chunk each record, embed the chunks in batches, and upsert them with an id derived from `source` + a content hash so re-runs are idempotent — see [chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) and [the RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb).

## 10. Practical B: CSV Q&A → chat-format JSONL for fine-tuning

OpenAI-style chat fine-tuning data is JSONL with one conversation per line:
`{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}`.
The converter below also does the unglamorous parts that decide dataset quality: drop rows with an empty question or answer, drop duplicate questions (after normalising case and spaces), and write to a temporary file then **rename** it into place, so a crash never leaves a half-written dataset behind (`Path.replace` is atomic on the same filesystem).

In [ ]:
faq_rows = [
    ("How long do refunds take?", "Refunds are issued within 30 days of delivery."),
    ("How long does shipping take?", "Standard shipping takes 3-5 business days."),
    ("how long do  refunds take?", "Within 30 days."),                         # duplicate after normalising
    ("Can I change my email?", ""),                                            # no answer yet
    ("Do you ship to Canada?", "Yes, to Canada and the US.\nExpress is US-only."),
]
faq_csv = WORK / "faq.csv"
with open(faq_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["question", "answer"])
    w.writerows(faq_rows)

SYSTEM = "You are Acme's support assistant. Answer in one or two sentences."

def csv_to_chat_jsonl(csv_path, out_path):
    seen, written, skipped = set(), 0, []
    tmp = out_path.with_suffix(".tmp")
    with open(csv_path, encoding="utf-8", newline="") as src, open(tmp, "w", encoding="utf-8") as dst:
        for row_no, row in enumerate(csv.DictReader(src), start=1):
            q, a = row["question"].strip(), row["answer"].strip()
            key = " ".join(q.lower().split())
            if not q or not a:
                skipped.append((row_no, "empty question or answer"))
                continue
            if key in seen:
                skipped.append((row_no, "duplicate question"))
                continue
            seen.add(key)
            example = {"messages": [{"role": "system", "content": SYSTEM},
                                    {"role": "user", "content": q},
                                    {"role": "assistant", "content": a}]}
            dst.write(json.dumps(example, ensure_ascii=False) + "\n")
            written += 1
    tmp.replace(out_path)                     # atomic rename into place
    return written, skipped

train = WORK / "train.jsonl"
written, skipped = csv_to_chat_jsonl(faq_csv, train)
print(f"wrote {written} examples; skipped {skipped}")

In [ ]:
def validate_chat_jsonl(path):
    problems = []
    for i, ex in enumerate(read_jsonl(path), start=1):
        roles = [m["role"] for m in ex["messages"]]
        if roles != ["system", "user", "assistant"]:
            problems.append((i, f"unexpected roles {roles}"))
        if any(not m["content"].strip() for m in ex["messages"]):
            problems.append((i, "empty content"))
    return problems

print(json.dumps(json.loads(train.read_text(encoding="utf-8").splitlines()[-1]), indent=2, ensure_ascii=False))
problems = validate_chat_jsonl(train)
print("problems:", problems)
assert written == 3 and [s[1] for s in skipped] == ["duplicate question", "empty question or answer"]
assert problems == [] and not train.with_suffix(".tmp").exists()

## Try it yourself

**1.** Count the total number of words across all records in `kb_records.jsonl` **without** loading the file into a list (hint: `sum` over a generator, reusing `read_jsonl`).

In [ ]:
# Solution — try it yourself first, then run this
total_words = sum(len(rec["text"].split()) for rec in read_jsonl(kb_jsonl))
print("total words:", total_words)
assert total_words == sum(len(d["text"].split()) for d in docs)

**2.** Find the two largest files anywhere under `KB`, with their sizes in bytes (hint: `rglob("*")`, `is_file()`, `stat().st_size`).

In [ ]:
# Solution — try it yourself first, then run this
sized = sorted(((p.stat().st_size, p.relative_to(KB).as_posix()) for p in KB.rglob("*") if p.is_file()), reverse=True)
for size, name in sized[:2]:
    print(f"{name:<22} {size} bytes")
assert sized[0][0] >= sized[1][0]

**3.** Save `{"id": 1, "created": datetime(2026, 1, 15, 9, 0, tzinfo=timezone.utc)}` to JSON with `default=`, load it back, and turn the string back into an equal `datetime` with `datetime.fromisoformat`.

In [ ]:
# Solution — try it yourself first, then run this
original = {"id": 1, "created": datetime(2026, 1, 15, 9, 0, tzinfo=timezone.utc)}
text = json.dumps(original, default=to_jsonable)
loaded = json.loads(text)
loaded["created"] = datetime.fromisoformat(loaded["created"])
print(text, "->", loaded["created"])
assert loaded == original

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)       # tidy up the scratch folder
print("scratch folder removed:", not WORK.exists())

## Say it in an interview

- **Paths:** "I use `pathlib` — `/` to join, `.suffix` / `.stem`, `rglob` to walk, `sorted()` for reproducible order. If a model or user can supply a path, I `resolve()` it and check `is_relative_to(base)`."
- **Encodings:** "Always `encoding='utf-8'` on every open, read and write. The default depends on the OS locale, and a wrong guess can silently corrupt text rather than raise."
- **Big files (dt01 / dt04):** "Stream: iterate the file or `csv.DictReader`, aggregate inside the loop, keep only running totals — memory is bounded by what I hold, not by file size. Generators chain into lazy pipelines; the trap is that they are single-use."
- **JSON:** "Keys become strings and tuples become lists. `ensure_ascii=False` for non-English text, `default=` for datetimes, `sort_keys=True` when I hash, `allow_nan=False` because `NaN` is not valid JSON."
- **JSONL:** "One object per line: streamable, appendable, one bad line loses one record. It is the format for fine-tuning, batch and eval files. I validate every line and report bad line numbers."
- **Datasets (dt14):** "Filter empties, deduplicate on a normalised key, validate the role structure, and write to a temp file then atomically rename."
- **Traps:** mode `"w"` truncates; forgetting `newline=""` with `csv`; `line.split(",")` on real CSV; `glob` order not guaranteed; `json.load` vs `json.loads` (file vs string).

## Next

- [11 · Modules, packages and a stdlib tour](11_modules_packages_and_stdlib_tour.ipynb) — `hashlib` content hashes, `re`, `collections` and more
- [07 · Comprehensions, iterators and generators](07_comprehensions_iterators_generators.ipynb) — generators in depth
- [Practical AI-engineer Python problems](../02_python_problems/04_practical_ai_engineer_python.ipynb)
- RAG ingestion: [chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) · [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb)
- Fine-tuning data: [when to fine-tune and data prep](../11_finetuning/01_when_to_fine_tune_and_data_prep.ipynb) · [the datasets library](../10_huggingface/03_datasets_library.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)